<a href="https://colab.research.google.com/github/CrUz-035/Tarea-1.-Git-y-documentaci-n-de-RNA/blob/main/Tarea_4_Francisco_Romerom_Mestiza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tarea: Reconocimiento facial

1. Entrenamiento de red convolucional con base da detos CelebA

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from tensorflow.keras.utils import image_dataset_from_directory

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import kagglehub

#Se instala la libreria donde se encuentra CelebA
!pip install kaggle

In [ ]:
import os
from google.colab import userdata
#Datos de la cuenta de Kaggle
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

#Se descarga el dataset de CelebA
path = kagglehub.dataset_download("jessicali9530/celeba-dataset")

Using Colab cache for faster access to the 'celeba-dataset' dataset.


In [ ]:
#Para conocer el contenido del dataset
contenido = os.listdir(path)

for i in contenido:
    print(i)

list_landmarks_align_celeba.csv
img_align_celeba
list_eval_partition.csv
list_attr_celeba.csv
list_bbox_celeba.csv


In [ ]:
#Para poder visualziar las categorias del dataset
atributos = os.path.join(path, 'list_attr_celeba.csv')
atributos_pd = pd.read_csv(atributos)
print(atributos_pd.head())

     image_id  5_o_Clock_Shadow  Arched_Eyebrows  Attractive  Bags_Under_Eyes  \
0  000001.jpg                -1                1           1               -1   
1  000002.jpg                -1               -1          -1                1   
2  000003.jpg                -1               -1          -1               -1   
3  000004.jpg                -1               -1           1               -1   
4  000005.jpg                -1                1           1               -1   

   Bald  Bangs  Big_Lips  Big_Nose  Black_Hair  ...  Sideburns  Smiling  \
0    -1     -1        -1        -1          -1  ...         -1        1   
1    -1     -1        -1         1          -1  ...         -1        1   
2    -1     -1         1        -1          -1  ...         -1       -1   
3    -1     -1        -1        -1          -1  ...         -1       -1   
4    -1     -1         1        -1          -1  ...         -1       -1   

   Straight_Hair  Wavy_Hair  Wearing_Earrings  Wearing_Hat  We

In [ ]:
print (atributos_pd.copy()[1:])

          image_id  5_o_Clock_Shadow  Arched_Eyebrows  Attractive  \
1       000002.jpg                -1               -1          -1   
2       000003.jpg                -1               -1          -1   
3       000004.jpg                -1               -1           1   
4       000005.jpg                -1                1           1   
5       000006.jpg                -1                1           1   
...            ...               ...              ...         ...   
202594  202595.jpg                -1               -1           1   
202595  202596.jpg                -1               -1          -1   
202596  202597.jpg                -1               -1          -1   
202597  202598.jpg                -1                1           1   
202598  202599.jpg                -1                1           1   

        Bags_Under_Eyes  Bald  Bangs  Big_Lips  Big_Nose  Black_Hair  ...  \
1                     1    -1     -1        -1         1          -1  ...   
2                

In [ ]:
from sklearn.model_selection import train_test_split #Para dividir las imágenes del entrenamiento.

In [ ]:
#Clasificando datos
#Trabajamos con una copia para no modificar el original
atr = atributos_pd.copy()
#Solo se ocupan 50000 imágenes
atr = atr.sample(n=50000, random_state=30)
# Normalizar los atributos de -1/1 a 0/1
for col in atr.columns[1:]: # Se ignora 'image_id'
    atr[col] = atr[col].apply(lambda x: 1 if x == 1 else 0)

# Construir la ruta de la carpeta de imágenes
image_base_dir = os.path.join(path, 'img_align_celeba', 'img_align_celeba')

# Relacionamos la imagen con su ruta
atr['image_path'] = atr['image_id'].apply(lambda x:
                                          os.path.join(image_base_dir, x))
# Filtrar filas para las cuales la imagen no existe
atr = atr[atr['image_path'].apply(os.path.exists)]


X = atr['image_path'] #Lo que el modelo recibirá para poder predecir
y = atr.drop(columns=['image_id', 'image_path']) #"Respuestas"
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3,
                                                  random_state=30)
#se separan datos para entrenamiento y para validación

#Se unen las direcciones de las imágemes con sus categorias.
df_train = pd.DataFrame({'image_path': X_train}).reset_index(drop=True)
df_train = pd.concat([df_train, y_train.reset_index(drop=True)], axis=1)

df_val = pd.DataFrame({'image_path': X_val}).reset_index(drop=True)
df_val = pd.concat([df_val, y_val.reset_index(drop=True)], axis=1)

train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
#El propósito de ImageDataGenerator es no saturar la ram.

#Datos de entrenamiento
train = train_datagen.flow_from_dataframe(
    dataframe=df_train, # Se emplea DataFrame construido
    x_col="image_path", # La columna de rutas de imagen
    y_col=df_train.columns[1:].tolist(), #Usa los datos de columnas de atributos
    target_size=(150, 150),
    batch_size=64,
    class_mode="raw", # Para clasificación multi-etiqueta
    shuffle=True # Mezclar los datos de entrenamiento
)

#Datos de validación
validation = val_datagen.flow_from_dataframe(
    dataframe=df_val,
    x_col="image_path",
    y_col=df_val.columns[1:].tolist(),
    target_size=(150, 150),
    batch_size=64,
    class_mode="raw",
    shuffle=False
)

Found 35000 validated image filenames.
Found 15000 validated image filenames.


In [ ]:
#Definir modelo
# Ajustar el input_shape del modelo al tamaño de las imágenes
inputs = keras.Input(shape=(150, 150, 3))
x = inputs

def block(x,base_filter=32, pooling=True):
  #Construye bloques de datos que se pueden volver a utilizar.
    residual = x #Se conserva dato para ocuparlo más tarde.
    #Se aplica convolucional
    x = layers.Conv2D(base_filter, 3, activation="relu", padding="same")(x)
    #Capa convolucional con el doble de filtros
    x = layers.Conv2D(2*base_filter, 3, activation="relu", padding="same")(x)
    if (pooling == True): #Se reducen dimensiones espaciales.
        x = layers.MaxPooling2D(2, padding="same")(x)
        residual = layers.Conv2D(2*base_filter, 1, strides=2)(residual)
    else:
      #Debe coincidir la dimensión con x.
        residual = layers.Conv2D(2*base_filter, 1)(residual)
    x = layers.add([x, residual]) #Se suma x original.
    return x

x = block(x,base_filter=32)
x = block(x,base_filter=48)
x = block(x,base_filter=64,pooling=False)

x = layers.GlobalAveragePooling2D()(x)
#Se reduce mapa de características a un promedio
#Drop Out
x = layers.Dropout(0.3)(x)

#Sigmoide para salida de datos, como son 40 clasificaciones, se especifíca.
outputs = layers.Dense(40, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)
model.summary() #Resumen

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 150, 150,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 150, 150,  │        896 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 150, 150,  │     18,496 │ conv2d_18[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 75, 75,    │          0 │ conv2d_19[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 75, 75,    │        256 │ input_layer_3[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 75, 75,    │          0 │ max_pooling2d_4[… │
│                     │ 64)               │            │ conv2d_20[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 75, 75,    │     27,696 │ add_6[0][0]       │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 75, 75,    │     41,568 │ conv2d_21[0][0]   │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 38, 38,    │          0 │ conv2d_22[0][0]   │
│ (MaxPooling2D)      │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 38, 38,    │      6,240 │ add_6[0][0]       │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 38, 38,    │          0 │ max_pooling2d_5[… │
│                     │ 96)               │            │ conv2d_23[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 38, 38,    │     55,360 │ add_7[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_25 (Conv2D)  │ (None, 38, 38,    │     73,856 │ conv2d_24[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 38, 38,    │     12,416 │ add_7[0][0]       │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 38, 38,    │          0 │ conv2d_25[0][0],  │
│                     │ 128)              │            │ conv2d_26[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ add_8[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ global_average_p

 Total params: 241,944 (945.09 KB)

 Trainable params: 241,944 (945.09 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.optimizers import Adam #Se importa el optimizador

In [ ]:
#Para guardar en dagshub
REPO_NAME= "Tarea_4"
REPO_OWNER = "CrUz-035"
USER_NAME = "CrUz-035"
!pip install -q mlflow #Instalamos Mlflow
!pip install mlflow --quiet

import mlflow
import os
from getpass import getpass
mlflow.tensorflow.autolog()

os.environ['MLFLOW_TRACKING_USERNAME'] = USER_NAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = getpass('Enter your DAGsHub access token or password: ')

mlflow.set_tracking_uri(f'https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow')

! pip3 install dagshub --upgrade #Para guardar exp en Dagshub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 25.1 MB/s eta 0:00:00
Enter your DAGsHub access token or password: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 12.6 MB

In [ ]:
#Entrenamiento del modelo

# MLflow setup
mlflow.set_experiment("Entrenamiento_rostro")
filepath = "Entrenamiento_rostro.h5"

earlystop = EarlyStopping(monitor='val_loss', mode='min',
                          restore_best_weights=True, patience=10, verbose=1)
checkpoint = ModelCheckpoint(filepath, monitor='val_loss',
                             verbose=1, save_best_only=True, mode='min')


with mlflow.start_run(run_name="experimento_2"):
  model.compile(loss='binary_crossentropy',
              optimizer=Adam(learning_rate=0.0001),
              metrics=[tf.keras.metrics.BinaryAccuracy(),
    tf.keras.metrics.AUC(),
    tf.keras.metrics.Precision(),
    tf.keras.metrics.Recall()]) #Métricas
  history = model.fit(train,
    epochs=40,
    validation_data=validation,
    callbacks=[earlystop, checkpoint],
    )
  mlflow.log_metric("final_val_binary_accuracy",
                  max(history.history["val_binary_accuracy"]))

  mlflow.log_metric("final_val_loss", min(history.history["val_loss"]))
  mlflow.log_artifact("Entrenamiento_rostro.h5")


2026/03/14 23:19:15 WARNING mlflow.tensorflow: Unrecognized dataset type <class 'keras.src.legacy.preprocessing.image.DataFrameIterator'>. Dataset logging skipped.
2026/03/14 23:19:15 WARNING mlflow.tensorflow: Unrecognized dataset type <class 'keras.src.legacy.preprocessing.image.DataFrameIterator'>. Dataset logging skipped.


Epoch 1/40
547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step - auc_1: 0.7115 - binary_accuracy: 0.7657 - loss: 0.5005 - precision_1: 0.4975 - recall_1: 0.3426
Epoch 1: val_loss improved from None to 0.41989, saving model to Entrenamiento_rostro.h5



Epoch 1: finished saving model to Entrenamiento_rostro.h5


547/547 ━━━━━━━━━━━━━━━━━━━━ 167s 290ms/step - auc_1: 0.7551 - binary_accuracy: 0.7925 - loss: 0.4596 - precision_1: 0.5678 - recall_1: 0.3290 - val_auc_1: 0.8049 - val_binary_accuracy: 0.8114 - val_loss: 0.4199 - val_precision_1: 0.6738 - val_recall_1: 0.3159
Epoch 2/40
547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step - auc_1: 0.7917 - binary_accuracy: 0.8074 - loss: 0.4299 - precision_1: 0.6368 - recall_1: 0.3330
Epoch 2: val_loss improved from 0.41989 to 0.41345, saving model to Entrenamiento_rostro.h5



Epoch 2: finished saving model to Entrenamiento_rostro.h5


...


Epoch 40: finished saving model to Entrenamiento_rostro.h5


547/547 ━━━━━━━━━━━━━━━━━━━━ 119s 218ms/step - auc_1: 0.9017 - binary_accuracy: 0.8625 - loss: 0.3114 - precision_1: 0.7736 - recall_1: 0.5504 - val_auc_1: 0.9086 - val_binary_accuracy: 0.8661 - val_loss: 0.3020 - val_precision_1: 0.7820 - val_recall_1: 0.5624
Restoring model weights from the end of the best epoch: 40.


2026/03/15 00:40:13 WARNING mlflow.tensorflow: Failed to infer model signature: could not sample data to infer model signature: '>=' not supported between instances of 'slice' and 'int'
2026/03/15 00:40:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/15 00:40:26 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during tensorflow autologging: BAD_REQUEST: Response: {'error_code': 'BAD_REQUEST'}


🏃 View run experimento_2 at: https://dagshub.com/CrUz-035/Tarea_4.mlflow/#/experiments/0/runs/f17ceea990ba4f278328e464cf7b91f4
🧪 View experiment at: https://dagshub.com/CrUz-035/Tarea_4.mlflow/#/experiments/0


FileNotFoundError: [Errno 2] No such file or directory: 'Entrenamiento_rostro_1.h5'

In [ ]:
from google.colab import files
files.download("Entrenamiento_rostro.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>